In [2]:
import os
import sys
import random
from collections import defaultdict, Counter

import pandas as pd

import eval as eval_module
import utils

In [2]:
# ========= paths =========
REF_PATH = '../data/OpenEA/D_W_15K_V2/'
RES_PATH = '../save/D_W_15K_V2.ttl'
CLEAN_RES_PATH = None

# ========= eval config =========
THRESHOLD = 0.1
PREFIX = "http://dbpedia.org/resource/"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [3]:
# ground truth
y_gold = eval_module.load_openea_ref(REF_PATH)
y_gold_set = set(y_gold)

print(f"Ground truth pairs: {len(y_gold)}")

# predicted scores + bilateral max assignment
sameAsscores, ent_max_assign = eval_module.load_ent_results(
    RES_PATH,
    prefix=PREFIX,
    threshold=THRESHOLD
)

print(f"Entities with candidate scores: {len(sameAsscores)}")
print(f"Entities in bilateral max assignment: {len(ent_max_assign)}")

Ground truth pairs: 15000
Entities with candidate scores: 14851
Entities in bilateral max assignment: 29124


In [4]:
eval_module.openea_eval(ent_max_assign, y_gold_set, save_path=None)

Precision: 0.9777
Recall: 0.9502
F1: 0.9638


1 Precision: 0.9541
Recall: 0.8415
F1: 0.8943

2 Precision: 0.9530
Recall: 0.8443
F1: 0.8954 (no double initialization)

3 Precision: 0.9538
Recall: 0.8443
F1: 0.8957 (double chunking + ent_max_assign calculated only once)

4 Precision: 0.9541
Recall: 0.8444
F1: 0.8959 (improve _match_entities_by_rules)

5 Precision: 0.9538
Recall: 0.8449
F1: 0.8960

6 Precision: 0.9397
Recall: 0.8117
F1: 0.8710 (literal matching buckets of lenght)

7 Precision: 0.9460
Recall: 0.8261
F1: 0.8820 (consider neighbor buckets)

8 Precision: 0.9408
Recall: 0.8153
F1: 0.8736 (prunning)

9 Precision: 0.9535
Recall: 0.8409
F1: 0.8937 (full search + prunning)

10 Precision: 0.9468
Recall: 0.8330
F1: 0.8863

In [21]:
kb1, kb2, gt_pairs = utils.load_openea(REF_PATH, attr=True)

print(f"KB1 facts: {len(kb1)}")
print(f"KB2 facts: {len(kb2)}")

   Loading OpenEA D_W_15K_V1...:   0%|          | 0/2 [00:00<?, ?it/s]

   Loading OpenEA D_W_15K_V1...: 100%|██████████| 2/2 [00:00<00:00,  3.50it/s]


KB1 facts: 213046
KB2 facts: 361984


In [8]:
REF_PATH = '../data/DBP15k/ja_en/'
RES_PATH = '../save/dbp-ja-en-w4_no_idf.ttl'

In [9]:
seed_pairs, ref_pairs = eval_module.load_dbp15k_ref(REF_PATH)
test_pairs = ref_pairs.copy()
ref_pairs.update(seed_pairs)
len(test_pairs), len(ref_pairs), len(seed_pairs)

(10500, 15000, 4500)

In [10]:
kb1_prefix = 'http://ja.dbpedia.org/resource/'
sameAsscores, ent_max_assign = eval_module.load_ent_results(RES_PATH, kb1_prefix)

In [11]:
# Ranking Based Metrics: refer to whole results
eval_module.dbp15k_eval(ref_pairs, sameAsscores)

Hit@1: 0.9619
Hit@10: 0.9745
MRR: 0.9657


In [3]:
# REF_PATH = '../data/OAEI/memoryalpha-stexpanded/'
# RES_PATH = '../save/memoryalpha-stexpanded_no_upper.ttl'

REF_PATH = '../data/OAEI/starwars-swtor/'
RES_PATH = '../save/starwars-swtor_hnsw.ttl'
# CLEAN_RES_PATH = '../save/memoryalpha-stexpanded-clean.ttl' starwars-swtor
CLEAN_RES_PATH = None

In [6]:
# prefix1 = 'http://dbkwik.webdatacommons.org/memory-alpha.wikia.com/'
# prefix2 = 'http://dbkwik.webdatacommons.org/stexpanded.wikia.com/'
# # starswars-swtor
prefix1 = 'http://dbkwik.webdatacommons.org/starwars.wikia.com/'
prefix2 = 'http://dbkwik.webdatacommons.org/swtor.wikia.com/'
threshold = 0.1

In [7]:
class_gt, property_gt, instance_gt = eval_module.load_oaei_ref(REF_PATH, prefix1)

In [8]:
len(class_gt), len(property_gt), len(instance_gt)

(15, 56, 1358)

In [9]:
y_pred_inst, y_pred_class, y_pred_similar, y_pred_sameAs = eval_module.load_full_results_oaei_kg_track(class_gt, instance_gt, property_gt, prefix1, RES_PATH)

In [10]:
eval_module.oaei_kg_eval(class_gt, instance_gt, property_gt,
                  y_pred_class, y_pred_inst, y_pred_sameAs, y_pred_similar,
                  prefix1, prefix2, threshold, save_path=CLEAN_RES_PATH)

**Instances: 
   Precision: 1.0000, Recall: 0.9116, F1: 0.9538
**Classes: 
   Precision: 1.0000, Recall: 0.8667, F1: 0.9286
**Properties: 
   Precision: 1.0000, Recall: 0.9643, F1: 0.9818
**Overall: 
   Precision: 1.0000, Recall: 0.9132, F1: 0.9546


### Addiction

In [22]:
def build_gold_map(y_gold):
    """
    Build a map from ground truth pairs.
    """
    return {e1: e2 for e1, e2 in y_gold}


def build_pred_map(ent_max_assign, prefix=None):
    """
    Build a map from predicted assignments.
    """
    pred_map = {}
    for e1, e2_dict in ent_max_assign.items():
        if prefix is not None and not e1.startswith(prefix):
            continue
        if not e2_dict:
            continue
        e2 = next(iter(e2_dict.keys()))
        pred_map[e1] = e2
    return pred_map


def build_pred_set_from_map(pred_map):
    return set(pred_map.items())


def get_entity_neighbors(graph, entity):
    """
    Get all neighbors of a given entity in the graph.
    """
    if entity not in graph.index:
        return []
    return list(graph.triplesWithSubject(entity))


def count_nonliteral_neighbors(graph, entity):
    """
    Count the number of non-literal neighbors for a given entity.
    """
    triples = get_entity_neighbors(graph, entity)
    count = 0
    for s, p, o in triples:
        if not utils.isLiteral(o):
            count += 1
    return count


def get_top_candidates(sameAsscores, e1, topk=10):
    """
    Get the top-k candidates for a given entity based on their scores.
    """
    cand_scores = sameAsscores.get(e1, {})
    return sorted(cand_scores.items(), key=lambda x: x[1], reverse=True)[:topk]


def extract_label_like_literals(graph, entity, max_items=10):
    """
    Extract literals that are likely to be labels (e.g., rdfs:label, foaf:name, etc.) for the given entity.
    """
    triples = get_entity_neighbors(graph, entity)
    literals = []
    for s, p, o in triples:
        if utils.isLiteral(o):
            literals.append((p, o))
    return literals[:max_items]


def safe_get_score(sameAsscores, e1, e2):
    return sameAsscores.get(e1, {}).get(e2, None)

In [ ]:
def classify_fp_errors(
    y_gold,
    ent_max_assign,
    sameAsscores,
    kb1,
    prefix=None,
    min_structure_neighbors=2,
    high_pred_score_for_literal=0.8,
    near_margin=0.05,
):
    gold_map = build_gold_map(y_gold)
    pred_map = build_pred_map(ent_max_assign, prefix=prefix)

    results = []
    summary = Counter()

    for e1, pred_e2 in pred_map.items():
        gold_e2 = gold_map.get(e1)

        # only analyze source entities that have gold answers 
        if gold_e2 is None:
            continue

        # false positive if predicted candidate is different from gold answer
        if pred_e2 == gold_e2:
            continue

        pred_score = safe_get_score(sameAsscores, e1, pred_e2)
        gold_score = safe_get_score(sameAsscores, e1, gold_e2)

        nonliteral_neighbors = count_nonliteral_neighbors(kb1, e1)
        top_candidates = get_top_candidates(sameAsscores, e1, topk=5)

        error_type = "unknown"
        note = ""

        # 1) gold candidate exists but lost in final competition
        if gold_score is not None:
            error_type = "competition_error"
            if pred_score is not None and abs(pred_score - gold_score) <= near_margin:
                note = "pred_score and gold_score are very close"
            else:
                note = "gold candidate exists but lost in final competition"

        # 2) gold candidate missing and source entity has few structural neighbors
        elif nonliteral_neighbors <= min_structure_neighbors:
            error_type = "structure_error"
            note = "gold candidate missing and source entity has few structural neighbors"

        # 3) wrong candidate has high score while gold candidate is absent
        elif pred_score is not None and pred_score >= high_pred_score_for_literal:
            error_type = "literal_error"
            note = "wrong candidate has high score while gold candidate is absent"

        # 4) other cases, likely due to relation / structural reasoning mismatch
        else:
            error_type = "relation_error"
            note = "likely propagated by relation / structural reasoning mismatch"

        results.append({
            "e1": e1,
            "pred_e2": pred_e2,
            "gold_e2": gold_e2,
            "pred_score": pred_score,
            "gold_score": gold_score,
            "nonliteral_neighbors": nonliteral_neighbors,
            "error_type": error_type,
            "note": note,
            "top_candidates": top_candidates,
        })

        summary[error_type] += 1

    df = pd.DataFrame(results)
    return df, summary

In [24]:
fp_df, fp_summary = classify_fp_errors(
    y_gold=y_gold,
    ent_max_assign=ent_max_assign,
    sameAsscores=sameAsscores,
    kb1=kb1,
    prefix=PREFIX,
    min_structure_neighbors=2,
    high_pred_score_for_literal=0.8,
    near_margin=0.05,
)

print("FP summary:")
for k, v in fp_summary.items():
    print(f"{k}: {v}")

fp_df.head(10)

FP summary:
literal_error: 31
relation_error: 98
competition_error: 392
structure_error: 104


,e1,pred_e2,gold_e2,pred_score,gold_score,nonliteral_neighbors,error_type,note,top_candidates
0,http://dbpedia.org/resource/Si_j'avais_au_moin...,http://www.wikidata.org/entity/Q1747247,http://www.wikidata.org/entity/Q2576321,0.968942,NaN,7,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q1747247, 0.9..."
1,http://dbpedia.org/resource/Plus_grandir,http://www.wikidata.org/entity/Q1747247,http://www.wikidata.org/entity/Q3392704,0.968942,NaN,7,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q1747247, 0.9..."
2,http://dbpedia.org/resource/Désenchantée,http://www.wikidata.org/entity/Q1747247,http://www.wikidata.org/entity/Q2576448,0.968942,NaN,6,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q1747247, 0.9..."
3,http://dbpedia.org/resource/C'est_une_belle_jo...,http://www.wikidata.org/entity/Q1747247,http://www.wikidata.org/entity/Q1424000,0.968942,NaN,8,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q1747247, 0.9..."
4,http://dbpedia.org/resource/Les_Mots_(song),http://www.wikidata.org/entity/Q1747247,http://www.wikidata.org/entity/Q1113167,0.968942,NaN,9,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q1747247, 0.9..."
5,http://dbpedia.org/resource/Holler_If_Ya_Hear_Me,http://www.wikidata.org/entity/Q847524,http://www.wikidata.org/entity/Q780306,0.125000,NaN,4,relation_error,likely propagated by relation / structural rea...,"[(http://www.wikidata.org/entity/Q847524, 0.12..."
6,http://dbpedia.org/resource/Yesterday's_Wine_(...,http://www.wikidata.org/entity/Q5583023,http://www.wikidata.org/entity/Q8052958,0.475069,NaN,7,relation_error,likely propagated by relation / structural rea...,"[(http://www.wikidata.org/entity/Q5583023, 0.4..."
7,http://dbpedia.org/resource/Osaka_Prefecture,http://www.wikidata.org/entity/Q35765,http://www.wikidata.org/entity/Q122723,0.986356,NaN,4,literal_error,wrong candidate has high score while gold cand...,"[(http://www.wikidata.org/entity/Q35765, 0.986..."
8,http://dbpedia.org/resource/Invasion_Quartet,http://www.wikidata.org/entity/Q6483655,http://www.wikidata.org/entity/Q6059575,0.116869,0.116869,1,competition_error,pred_score and gold_score are very close,"[(http://www.wikidata.org/entity/Q3955147, 0.1..."
9,http://dbpedia.org/resource/Basics_(Star_Trek:...,http://www.wikidata.org/entity/Q7315598,http://www.wikidata.org/entity/Q4867063,0.250000,0.250000,3,competition_error,pred_score and gold_score are very close,"[(http://www.wikidata.org/entity/Q7315598, 0.2..."


In [25]:
def classify_fn_errors(
    y_gold,
    ent_max_assign,
    sameAsscores,
    kb1,
    prefix=None,
    min_structure_neighbors=2,
):
    gold_map = build_gold_map(y_gold)
    pred_map = build_pred_map(ent_max_assign, prefix=prefix)

    results = []
    summary = Counter()

    for e1, gold_e2 in gold_map.items():
        if prefix is not None and not e1.startswith(prefix):
            continue

        pred_e2 = pred_map.get(e1, None)
        if pred_e2 == gold_e2:
            continue # only analyze false negatives

        gold_score = safe_get_score(sameAsscores, e1, gold_e2)
        nonliteral_neighbors = count_nonliteral_neighbors(kb1, e1)
        top_candidates = get_top_candidates(sameAsscores, e1, topk=5)

        if gold_score is not None:
            error_type = "competition_or_threshold_error"
            note = "gold candidate exists in score table but not in final assignment"
        elif nonliteral_neighbors <= min_structure_neighbors:
            error_type = "structure_error"
            note = "gold candidate absent and source entity has few structural neighbors"
        else:
            error_type = "literal_or_relation_missing_error"
            note = "gold candidate absent; may be due to literal miss or relation propagation failure"

        results.append({
            "e1": e1,
            "pred_e2": pred_e2,
            "gold_e2": gold_e2,
            "gold_score": gold_score,
            "nonliteral_neighbors": nonliteral_neighbors,
            "error_type": error_type,
            "note": note,
            "top_candidates": top_candidates,
        })

        summary[error_type] += 1

    df = pd.DataFrame(results)
    return df, summary

In [26]:
fn_df, fn_summary = classify_fn_errors(
    y_gold=y_gold,
    ent_max_assign=ent_max_assign,
    sameAsscores=sameAsscores,
    kb1=kb1,
    prefix=PREFIX,
    min_structure_neighbors=2,
)

print("FN summary:")
for k, v in fn_summary.items():
    print(f"{k}: {v}")

fn_df.head(10)

FN summary:
literal_or_relation_missing_error: 693
structure_error: 1068
competition_or_threshold_error: 574


,e1,pred_e2,gold_e2,gold_score,nonliteral_neighbors,error_type,note,top_candidates
0,"http://dbpedia.org/resource/Wow,_The_Kid_Gang_...",http://www.wikidata.org/entity/Q958664,http://www.wikidata.org/entity/Q10921857,NaN,3,literal_or_relation_missing_error,gold candidate absent; may be due to literal m...,"[(http://www.wikidata.org/entity/Q958664, 0.33..."
1,http://dbpedia.org/resource/Yebawmi,None,http://www.wikidata.org/entity/Q8051023,NaN,2,structure_error,gold candidate absent and source entity has fe...,[]
2,http://dbpedia.org/resource/Nail_(album),None,http://www.wikidata.org/entity/Q3869958,NaN,2,structure_error,gold candidate absent and source entity has fe...,[]
3,http://dbpedia.org/resource/Gärstenhörner,http://www.wikidata.org/entity/Q3647563,http://www.wikidata.org/entity/Q920863,0.25,2,competition_or_threshold_error,gold candidate exists in score table but not i...,"[(http://www.wikidata.org/entity/Q3647563, 0.2..."
4,http://dbpedia.org/resource/Christmas_Party_(s...,None,http://www.wikidata.org/entity/Q15091102,NaN,3,literal_or_relation_missing_error,gold candidate absent; may be due to literal m...,[]
5,http://dbpedia.org/resource/How_High_Is_Up%3F,http://www.wikidata.org/entity/Q7044026,http://www.wikidata.org/entity/Q5917759,0.50,7,competition_or_threshold_error,gold candidate exists in score table but not i...,"[(http://www.wikidata.org/entity/Q7044026, 0.5..."
6,"http://dbpedia.org/resource/B-Sides,_Remixes_a...",None,http://www.wikidata.org/entity/Q4833694,NaN,2,structure_error,gold candidate absent and source entity has fe...,[]
7,http://dbpedia.org/resource/Cabal_(novella),None,http://www.wikidata.org/entity/Q1990342,NaN,2,structure_error,gold candidate absent and source entity has fe...,[]
8,http://dbpedia.org/resource/Say_Say_Say_(Waiti...,None,http://www.wikidata.org/entity/Q3951187,NaN,3,literal_or_relation_missing_error,gold candidate absent; may be due to literal m...,"[(http://www.wikidata.org/entity/Q905780, 0.14..."
9,http://dbpedia.org/resource/Rafsanjan_County,None,http://www.wikidata.org/entity/Q1287811,NaN,2,structure_error,gold candidate absent and source entity has fe...,[]
